In [ ]:
# %matplotlib widget
%matplotlib ipympl
# %matplotlib notebook

# Imports

In [ ]:
from loguru import logger
from os import getcwd
from os.path import join
from unittest import TestCase
from xarray import Dataset
from collections import OrderedDict
from matplotlib import ticker

import os
import matplotlib
import matplotlib.pyplot as pl
import numpy as np
import pandas as pd
import astropy.units as uu
import arviz as az
import xarray as xr
import pprint
import scipy as sp
import json

from corner import corner

In [ ]:
import lisa.posterior.core.posterior as cpost
import lisa.emcee_tools.emcee_tools as et
import lisa.tools.stats.distribution_anali as da
import lisa.posterior.exoplanet.model.convert as cv
import lisa.posterior.exoplanet.exploration_analysis_tools.secondary_parameters as secpar


from lisa.tools.chain_interpreter import ChainsInterpret
from lisa.explore_analyze.misc import get_def_output_folders
from lisa.explore_analyze.plot import hist_lnprob
from lisa.explore_analyze.posterior_plots import posterior_plot

In [ ]:
import importlib
import lisa.explore_analyze.posterior_plots
importlib.reload(lisa.explore_analyze.posterior_plots)

# Global parameters

In [ ]:
obj_name = "WASP-95"  # Change

run_name = "initrun"
# extension_exploration = "_fuprun"  # extension of your exploration (Needs to be the same than in you script_mcmcexploration)

In [ ]:
output_folders = get_def_output_folders(run_folder=getcwd())  # Folder for the outputs

In [ ]:
save_plots = True  # Save all the plots.
extension_outputs = f"_{run_name}"  # extension of this chain analysis (will be added to the ouput files).

# Load backend

In [ ]:
import emcee
backend_filename = f"{obj_name}_Emcee.h5"
backend_filepath = join(output_folders["pickles_explore"], backend_filename)
backend = emcee.backends.HDFBackend(backend_filepath, read_only=True, name=run_name)
nwalkers, nparams = backend.shape
print(f"The chain stored in {backend_filename} has {nwalkers} walkers and {nparams} parameters.")

# Assess burnin

In [ ]:
thin_burnin = 100
lp = backend.get_log_prob(thin=thin_burnin)
nstep = lp.shape[0]
min_log10 = np.sign(lp.min()) * np.log10(abs(lp.min()))
max_log10 = np.sign(lp.max()) * np.log10(abs(lp.max()))
if (max_log10 - min_log10) > 1.5:
    log_scale = True
else:
    log_scale = False
fig, ax = pl.subplots(figsize=(10, 4), constrained_layout=True)
ax.set_xlabel("Steps")
ax.set_ylabel("ln posterior")
for walker_i in range(nwalkers):
    line = ax.plot(np.arange(nstep) * thin_burnin, lp[..., walker_i], alpha=0.5)
if log_scale:
    ax.set_yscale('symlog')

In [ ]:
pl.savefig(join(output_folders["plots"], f"trace_raw_lp{extension_outputs}.png"))


In [ ]:
burnin = 15000
print(f"Burnin chosen: {np.round(burnin)}")
print(f"Number of samples after burnin: {nstep * thin_burnin - burnin}")

# Assess thining and convergence: Autocorrelation time

In [ ]:
thin_autocorr = 100
tau = backend.get_autocorr_time(discard=burnin, tol=50, thin=thin_autocorr, quiet=True)
print(f"Mean autocorrelation time: {np.mean(tau)}")

In [ ]:
thin = int(0.1 * np.min(tau))
print(f"Thin chosen: {thin}")
print(f"Number of samples after burnin and thinning: {(nstep * thin_burnin - burnin) // thin} per walker")

# Save burnin and thin

In [ ]:
# Create a dictionary
analysis_params = {'burnin': burnin, 'thin': thin}

# Open a file in write mode
with open(join(output_folders["pickles_analyze"], f'analysis_params{extension_outputs}.json'), 'w') as json_file:
    # Write the dictionary to the file as a JSON string
    json_file.write(json.dumps(analysis_params))

# Load Inference data

In [ ]:
# Load parameter names
with open(join(output_folders["pickles_explore"], f"{obj_name}_param_names.json"), "r") as file:
    l_param_name = json.load(file)
l_param_name

In [ ]:
from arviz import InferenceData, dict_to_dataset
import emcee
# blobs_dict = self.blobs_to_dict()
# Compute the parameters posteriors
chain = backend.get_chain(flat=False, thin=thin, discard=burnin)
posterior = dict_to_dataset(data={var_name: (chain[(..., idx_var)].swapaxes(0, 1)) for idx_var, var_name in enumerate(l_param_name)},
                            library=emcee,
                            # coords=["draw", "walker"],
                            # dims=self.dims,
                            # index_origin=self.index_origin,
                            )
del chain
# Compute the log posterior, log prior and log likelihood
blob_dict = {"sample_stats": {}}
blob_dict["sample_stats"]["lp"] = backend.get_log_prob(flat=False, thin=thin, discard=burnin).swapaxes(0, 1)
blobs = backend.get_blobs(flat=False, thin=thin, discard=burnin).swapaxes(0, 1)
blob_dict["sample_stats"]["lprior"] = blobs["log_prior"]
blob_dict["sample_stats"]["llike"] = blobs["log_likelihood"]
del blobs
# Compute the blobs
for key, values in blob_dict.items():
    blob_dict[key] = dict_to_dataset(values, library=emcee)
# obs_const_dict = self.args_to_xarray()
infdata = InferenceData(posterior=posterior, 
                        **blob_dict,
                        # **obs_const_dict
                        )
del posterior, blob_dict
infdata

# Trace plots

## Posterior trace plot

In [ ]:
az.plot_trace(infdata.sample_stats.lp, kind="trace", compact=False, backend_kwargs={"constrained_layout": True})
pl.savefig(join(output_folders["plots"], f"traceNdist_burnin{burnin}thin{thin}_lp{extension_outputs}.png"))


## Parameters trace plots

In [ ]:
az.plot_trace(infdata, kind="trace",compact=False, backend_kwargs={"constrained_layout": True})
pl.savefig(join(output_folders["plots"], f"traceNdist_burnin{burnin}thin{thin}_param{extension_outputs}.png"))
pl.show()

In [ ]:
pl.close("all")

# Summary statistics

In [ ]:
summary = az.summary(infdata, round_to="none")
summary

In [ ]:
summary.to_csv(join(output_folders["tables"], f"summary_stats{extension_outputs}.tsv"), sep="\t")

## Rhat

WARNING: Rhat assume that the chains are independent while it's not the case for the walker of an Ensemble Sampler

In [ ]:
rhat_thresold = 1.1
if sum(summary["r_hat"] > rhat_thresold) > 0:
    logger.warning(f"Rhat values are above the thresold for convergence ({rhat_thresold}) for the following parameters:\n{summary.index[summary['r_hat'] > rhat_thresold].values}.\nThe chains have not converged.")
else:
    logger.info(f"According to the Rhat criteria (<{rhat_thresold}) all parameters chains are converged")

# Compute best-fit parameters (main)

In [ ]:
prob_1sigma = 0.682689492137	
prob_3sigma = 0.997300203937
probs_names = ["3%", "68%", "99.7%"]
probs_values = [0.03, 0.68, 0.997]

### HDIs

In [ ]:
hdis = {}
coords = {"hdi": ["mean_hdi", "mean_all", "median_hdi", "median_all", "lower", "higher", 
                  "delta_lowermeanhdi", "delta_highermeanhdi", "delta_lowermeanall", "delta_highermeanall",
                  "delta_lowermedianhdi", "delta_higheredianhdi", "delta_lowermedianall", "delta_higheredianall",]}
attrs = {"description": "lower, higher boundary of the hdi interval and the mean and median of the posterior within the hdi interval"}
data_vars = {}
for key, prob in zip(probs_names, probs_values):
    hdi_prob = az.hdi(infdata, hdi_prob=prob, multimodal=False)
    infdata_where_hdi_prob = infdata.posterior.where(infdata.posterior.data_vars > hdi_prob.sel(hdi="lower")).where(infdata.posterior.data_vars < hdi_prob.sel(hdi="higher"), drop=True)
    for var in infdata.posterior.keys():
        data_vars[var] = (["hdi"], [infdata_where_hdi_prob.mean()[var].values, 
                                    infdata.posterior[var].mean().values,
                                    infdata_where_hdi_prob.median()[var].values,
                                    infdata.posterior[var].median().values,
                                    float(hdi_prob[var].sel(hdi = "lower").values), 
                                    float(hdi_prob[var].sel(hdi = "higher").values), 
                                    float(hdi_prob[var].sel(hdi = "lower").values) - infdata_where_hdi_prob.mean()[var].values, 
                                    float(hdi_prob[var].sel(hdi = "higher").values) - infdata_where_hdi_prob.mean()[var].values,
                                    float(hdi_prob[var].sel(hdi = "lower").values) - infdata.posterior[var].mean().values, 
                                    float(hdi_prob[var].sel(hdi = "higher").values) - infdata.posterior[var].mean().values,
                                    float(hdi_prob[var].sel(hdi = "lower").values) - infdata_where_hdi_prob.median()[var].values, 
                                    float(hdi_prob[var].sel(hdi = "higher").values) - infdata_where_hdi_prob.median()[var].values,
                                    float(hdi_prob[var].sel(hdi = "lower").values) - infdata.posterior[var].median().values, 
                                    float(hdi_prob[var].sel(hdi = "higher").values) - infdata.posterior[var].median().values,
                                    ])
    hdis[key] = xr.Dataset(data_vars=data_vars, coords=coords, attrs=attrs)

In [ ]:
# Save to file
for prob_name in probs_names:
    hdis[prob_name].to_dataframe().transpose().to_csv(join(output_folders["tables"], f"hdi{prob_name}{extension_outputs}.tsv"), sep="\t")

### etis_sec

In [ ]:
etis_sec = {}
coords = {"eti": ["mean_eti", "mean_all", "median", "lower", "higher", "delta_lowermeaneti", "delta_highermeaneti", "delta_lowermeanall", "delta_highermeanall", "delta_lowermedian", "delta_highermedian"]}
attrs = {"description": "lower, higher boundary of the eti interval and the mean and median of the posterior within the eti interval"}
data_vars = {}
for key, prob in zip(probs_names, probs_values):
    infdata_where_hdi_prob = infdata.posterior.where(infdata.posterior.data_vars > hdi_prob.sel(hdi="lower"), drop=True).where(infdata.posterior.data_vars < hdi_prob.sel(hdi="higher"), drop=True)
    for var in infdata.posterior.keys():
        lower_percentile = 50 * (1 - prob)
        higher_percentile = 50 * (1 + prob)
        lower = sp.stats.scoreatpercentile(infdata.posterior[var].values, lower_percentile)
        higher = sp.stats.scoreatpercentile(infdata.posterior[var].values, higher_percentile)
        data_var_where_eti_prob = infdata.posterior[var].where(infdata.posterior[var] > lower).where(infdata.posterior[var] < higher)
        data_vars[var] = (["eti"], [data_var_where_eti_prob.mean().values, 
                                    infdata.posterior[var].mean().values,
                                    infdata.posterior[var].median().values,
                                    lower, 
                                    higher, 
                                    lower - data_var_where_eti_prob.mean().values, 
                                    higher - data_var_where_eti_prob.mean().values,
                                    lower - infdata.posterior[var].mean().values, 
                                    higher - infdata.posterior[var].mean().values,
                                    lower - data_var_where_eti_prob.median().values, 
                                    higher - data_var_where_eti_prob.median().values,
                                    ])
    etis_sec[key] = xr.Dataset(data_vars=data_vars, coords=coords, attrs=attrs)

In [ ]:
# Save to file
for prob_name in probs_names:
    etis_sec[prob_name].to_dataframe().transpose().to_csv(join(output_folders["tables"], f"eti{prob_name}{extension_outputs}.tsv"), sep="\t")

### Fitted values

In [ ]:
coords = {"fit": ["mean3_hdi", "delta_lower68mean3_hdi", "delta_higher68mean3_hdi",
                  "delta_lower99.7mean3_hdi", "delta_higher99.7mean3_hdi", 
                  "median", "delta_lower68median_eti", "delta_higher68median_eti",
                  "lower99.7_hdi", "higher99.7_hdi",
                  "lower99.7_eti", "higher99.7_eti"
                  ]}
attrs = {"description": "Infered values and errors with different criteria"}
data_vars = {}
for var in infdata.posterior.keys():
    data_vars[var] = (["fit"], [hdis["3%"][var].sel(hdi="mean_hdi").values, 
                                hdis["68%"][var].sel(hdi="lower").values - hdis["3%"][var].sel(hdi="mean_hdi").values,
                                hdis["68%"][var].sel(hdi="higher").values - hdis["3%"][var].sel(hdi="mean_hdi").values,
                                hdis["99.7%"][var].sel(hdi="lower").values - hdis["3%"][var].sel(hdi="mean_hdi").values,
                                hdis["99.7%"][var].sel(hdi="higher").values - hdis["3%"][var].sel(hdi="mean_hdi").values,
                                etis_sec["68%"][var].sel(eti="median").values,
                                etis_sec["68%"][var].sel(eti="lower").values - etis_sec["68%"][var].sel(eti="median").values,
                                etis_sec["68%"][var].sel(eti="higher").values - etis_sec["68%"][var].sel(eti="median").values,
                                hdis["99.7%"][var].sel(hdi="lower").values, 
                                hdis["99.7%"][var].sel(hdi="higher").values,
                                etis_sec["99.7%"][var].sel(eti="lower").values, 
                                etis_sec["99.7%"][var].sel(eti="higher").values,
                                ]
                      )
fitted_values = xr.Dataset(data_vars=data_vars, coords=coords, attrs=attrs)
fitted_values

In [ ]:
# Save to file
fitted_values.to_dataframe().transpose().to_csv(join(output_folders["tables"], f"fittedvalues{extension_outputs}.tsv"), sep="\t")

### df_fittedvals

In [ ]:
# created the df_fitted_values and save it for the plot and the re-run
value = "mean3_hdi"
sigma_m = "delta_lower68mean3_hdi"
sigma_p = "delta_higher68mean3_hdi"
df_fittedval = pd.DataFrame(index=list(fitted_values.keys()), 
                            data={'value': [float(fitted_values[var].sel(fit=value).values) for var in fitted_values.keys()], 
                                  'sigma-': [float(fitted_values[var].sel(fit=sigma_m).values) for var in fitted_values.keys()],
                                  'sigma+': [float(fitted_values[var].sel(fit=sigma_p).values) for var in fitted_values.keys()],
                                  }
                            )
df_fittedval

In [ ]:
# save to pickle
et.save_chain_analysis(obj_name, extension_analysis=extension_outputs, fitted_values={"array": np.array([fitted_values[var].sel(fit=value).values for var in fitted_values.keys()]), "l_param": list(fitted_values.keys())},
                       df_fittedval=df_fittedval, folder=output_folders["pickles_analyze"])

# Corner plots (main)

In [ ]:
thin_chain = 200
corner(infdata.posterior.thin(indexers={"draw": thin_chain}), truths=[fitted_values[var].sel(fit="mean3_hdi").values for var in fitted_values])
pl.savefig(join(output_folders["plots"], f"corner{extension_outputs}.png"))
pl.show()

# Posterior plots (main)

## Arviz posterior plot

In [ ]:
az.plot_posterior(infdata.posterior, hdi_prob=0.68, point_estimate='mode')
pl.savefig(join(output_folders["plots"], f"posterior_arviz{extension_outputs}.png"))


## My posteriors

In [ ]:
best_value = "mean3_hdi"
shaded_lower1s = "delta_lower68mean3_hdi"
shaded_higher1s = "delta_higher68mean3_hdi"
shaded_lower3s = "delta_lower99.7mean3_hdi"
shaded_higher3s = "delta_higher99.7mean3_hdi"

# For multiple posterior plots
# nrows, ncols = az.plots.plot_utils.default_grid(n_items=len(infdata.posterior), grid=None, max_cols=4, min_cols=2)
# figsize, ax_labelsize, titlesize, xt_labelsize, linewidth, markersize = az.plots.plot_utils._scale_fig_size(figsize=(10,10), textsize=8, rows=1, cols=1)
# l_param = infdata.posterior.keys()

l_param = ["b_A", ]
paramname4label = {"b_A": "PC Amplitude"}
factor = {"b_A": 1e6}
unit = {"b_A": "ppm"}

nrows = 1
ncols = len(l_param)
figsize = (4, 3)

fig, axes = pl.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, squeeze=False)
for ii, var in enumerate(l_param):
    i_col = ii % ncols
    i_row = ii // ncols
    ax = axes[i_row, i_col] 
    var_factor = factor[var] if var in factor else 1
    var_xlabel = paramname4label[var] if var in paramname4label else var
    if var in unit:
        var_xlabel += f" [{unit[var]}]"
    show_ylabel = True if i_col == 0 else False
    posterior_plot(samples=infdata.posterior[var] * var_factor, 
                   values_wkwargs=[(fitted_values[var].sel(fit=best_value).values * var_factor,{"color": "C0"})],
                   intervals_wkwargs=[(((fitted_values[var].sel(fit=best_value).values + fitted_values[var].sel(fit=shaded_lower3s).values) * var_factor, 
                                        (fitted_values[var].sel(fit=best_value).values + fitted_values[var].sel(fit=shaded_higher3s).values) * var_factor),
                                       {"color": "C0", 'alpha': 0.2}
                                       ),
                                      (((fitted_values[var].sel(fit=best_value).values + fitted_values[var].sel(fit=shaded_lower1s).values) * var_factor,  
                                        (fitted_values[var].sel(fit=best_value).values + fitted_values[var].sel(fit=shaded_higher1s).values) * var_factor),
                                       {"color": "C0", 'alpha': 0.5}
                                       ),
                                      ], 
                   xlabel=var_xlabel, 
                   show_ylabel=show_ylabel, ax=ax)

# Secondary parameters

## Compute secondary parameters

In [ ]:
from numpy.random import normal

In [ ]:
timestamp_stellar_parameters = "20250728_1409"
stellar_parameters_directory = '/Users/olivier/Projects/CHEOPS/Axis2/occultation_3HJ/lisa_analysis/Stellar_parameters/'
filepath = os.path.join(stellar_parameters_directory, obj_name, f"stellar_params_{obj_name}_{timestamp_stellar_parameters}.json")
with open(filepath, "r") as f:
    stellar_parameters = json.load(f)
print(stellar_parameters)

{'target': 'HD 149026', 'gaia_id': '1331356474971716992', 'ra': '247.62341', 'dec': '38.34730833', 'Vmag': '8.14', 'ts3_flag': 1.0, 'teff': 6162.0, 'e_teff': 41.0, 'logg': 4.37, 'e_logg': 0.1, 'feh': 0.36, 'e_feh': 0.05, 'vmic': '1.41', 'e_vmic': '0.07', 'vmac': None, 'e_vmac': None, 'vsini': '6', 'e_vsini': '0.5', 'r_vsini': 'Bonomo+ 2017', 'radius': 1.451, 'e_radius': 0.012, 'eBV': 0.034, 'mass': 1.315, 'eLow_mass': 0.051, 'eUpp_mass': 0.047, 'eMax_mass': 0.051, 'age': 2.4, 'eLow_age': 0.5, 'eUpp_age': 0.5, 'eMax_age': 0.5, 'mgh': '0.29', 'e_mgh': '0.05', 'sih': '0.33', 'e_sih': '0.04', 'logRhk': -4.93, 'r_logRhk': 'Boro Saikia+ 2018', 'prot': None, 'r_prot': None, 'Update Spec': '17 June 2020', 'Update R': '16 Apr 2025', 'Update MA': '12-May-2025', 'qc_flag': 'g', 'remarks': 'Gaia DR3 BMA', 'Spectra Availability': None}


In [ ]:
secpar_dict = OrderedDict()

secpar_dict['A_M'] = {'function': normal, 'kwargs': {'loc': stellar_parameters['mass'], 'scale': (stellar_parameters['eLow_mass'] + stellar_parameters['eUpp_mass']) / 2, 'size': 'shape(nwalker,niter)'}, 'l_param_name': None}
secpar_dict['A_R'] = {'function': normal, 'kwargs': {'loc': stellar_parameters['radius'], 'scale': stellar_parameters['e_radius'], 'size': 'shape(nwalker,niter)'}, 'l_param_name': None}
secpar_dict['A_Teff'] = {'function': normal, 'kwargs': {'loc': stellar_parameters['teff'], 'scale': stellar_parameters['e_teff'], 'size': 'shape(nwalker,niter)'}, 'l_param_name': None}
for planet in ['b', ]:
      secpar_dict[f'{planet}_ecc'] = {'function': cv.getecc, 'kwargs': None, 'l_param_name': [f'{planet}_ecosw', f'{planet}_esinw']}
      secpar_dict[f'{planet}_omega'] = {'function': cv.getomega_deg, 'kwargs': None, 'l_param_name': [f'{planet}_ecosw', f'{planet}_esinw']}
      secpar_dict[f'{planet}_inc'] = {'function': cv.getinc, 'kwargs': None, 'l_param_name': [f'{planet}_cosinc', ]}
      secpar_dict[f'{planet}_aR'] = {'function': cv.getaoverr, 'kwargs': None, 'l_param_name': [f'{planet}_P', 'A_rho', f'{planet}_ecc', f'{planet}_omega']}
      secpar_dict[f'{planet}_b'] = {'function': cv.getb, 'kwargs': None, 'l_param_name': [f'{planet}_inc', f'{planet}_aR', f'{planet}_ecc', f'{planet}_omega']}
      secpar_dict[f'{planet}_R'] = {'function': cv.getRp, 'kwargs': None, 'l_param_name': [f'{planet}_Rrat', 'A_R']}
      secpar_dict[f'{planet}_Frat'] = {'function': cv.getFrat_sincos, 'kwargs': {'Foffset': 0}, 'l_param_name': [f'{planet}_A']}
      secpar_dict[f'{planet}_Phioffset'] = {'function': cv.getPhioffset_sincos, 'kwargs': {'sincos': 'cos'}, 'l_param_name': [f'{planet}_Phi', ]}
      # secpar_dict[f'{planet}_M'] = {'function': cv.getMpsininc, 'kwargs': {'Kfact': 1}, 'l_param_name': [f'{planet}_P', f'{planet}_K', 'A_M', f'{planet}_ecc']}

In [ ]:

post_instance = cpost.Posterior()
post_instance.configure_posterior(path_config_file="config_file.py")
post_instance.create_allfunctions()

In [ ]:
posterior_sec = secpar.get_secondary_xrdataset(infdata=infdata, sec_params=secpar_dict, model=post_instance.model)
posterior_sec

## Compute best-fit parameters

### HDIs

In [ ]:
hdis_sec = {}
coords = {"hdi": ["mean_hdi", "mean_all", "median_hdi", "median_all", "lower", "higher", 
                  "delta_lowermeanhdi", "delta_highermeanhdi", "delta_lowermeanall", "delta_highermeanall",
                  "delta_lowermedianhdi", "delta_higheredianhdi", "delta_lowermedianall", "delta_higheredianall",]}
attrs = {"description": "lower, higher boundary of the hdi interval and the mean and median of the posterior within the hdi interval"}
data_vars = {}
for key, prob in zip(probs_names, probs_values):
    hdi_prob = az.hdi(posterior_sec, hdi_prob=prob, multimodal=False)
    infdata_where_hdi_prob = posterior_sec.where(posterior_sec.data_vars > hdi_prob.sel(hdi="lower")).where(posterior_sec.data_vars < hdi_prob.sel(hdi="higher"), drop=True)
    for var in posterior_sec.keys():
        data_vars[var] = (["hdi"], [infdata_where_hdi_prob.mean()[var].values, 
                                    posterior_sec[var].mean().values,
                                    infdata_where_hdi_prob.median()[var].values,
                                    posterior_sec[var].median().values,
                                    float(hdi_prob[var].sel(hdi = "lower").values), 
                                    float(hdi_prob[var].sel(hdi = "higher").values), 
                                    float(hdi_prob[var].sel(hdi = "lower").values) - infdata_where_hdi_prob.mean()[var].values, 
                                    float(hdi_prob[var].sel(hdi = "higher").values) - infdata_where_hdi_prob.mean()[var].values,
                                    float(hdi_prob[var].sel(hdi = "lower").values) - posterior_sec[var].mean().values, 
                                    float(hdi_prob[var].sel(hdi = "higher").values) - posterior_sec[var].mean().values,
                                    float(hdi_prob[var].sel(hdi = "lower").values) - infdata_where_hdi_prob.median()[var].values, 
                                    float(hdi_prob[var].sel(hdi = "higher").values) - infdata_where_hdi_prob.median()[var].values,
                                    float(hdi_prob[var].sel(hdi = "lower").values) - posterior_sec[var].median().values, 
                                    float(hdi_prob[var].sel(hdi = "higher").values) - posterior_sec[var].median().values,
                                    ])
    hdis_sec[key] = xr.Dataset(data_vars=data_vars, coords=coords, attrs=attrs)

In [ ]:
# Save to file
for prob_name in probs_names:
    hdis_sec[prob_name].to_dataframe().transpose().to_csv(join(output_folders["tables"], f"hdi{prob_name}_secpar{extension_outputs}.tsv"), sep="\t")

### ETIs

In [ ]:
etis_sec = {}
coords = {"eti": ["mean_eti", "mean_all", "median", "lower", "higher", "delta_lowermeaneti", "delta_highermeaneti", "delta_lowermeanall", "delta_highermeanall", "delta_lowermedian", "delta_highermedian"]}
attrs = {"description": "lower, higher boundary of the eti interval and the mean and median of the posterior within the eti interval"}
data_vars = {}
for key, prob in zip(probs_names, probs_values):
    infdata_where_hdi_prob = posterior_sec.where(posterior_sec.data_vars > hdi_prob.sel(hdi="lower"), drop=True).where(posterior_sec.data_vars < hdi_prob.sel(hdi="higher"), drop=True)
    for var in posterior_sec.keys():
        lower_percentile = 50 * (1 - prob)
        higher_percentile = 50 * (1 + prob)
        lower = sp.stats.scoreatpercentile(posterior_sec[var].values, lower_percentile)
        higher = sp.stats.scoreatpercentile(posterior_sec[var].values, higher_percentile)
        data_var_where_eti_prob = posterior_sec[var].where(posterior_sec[var] > lower).where(posterior_sec[var] < higher)
        data_vars[var] = (["eti"], [data_var_where_eti_prob.mean().values, 
                                    posterior_sec[var].mean().values,
                                    posterior_sec[var].median().values,
                                    lower, 
                                    higher, 
                                    lower - data_var_where_eti_prob.mean().values, 
                                    higher - data_var_where_eti_prob.mean().values,
                                    lower - posterior_sec[var].mean().values, 
                                    higher - posterior_sec[var].mean().values,
                                    lower - data_var_where_eti_prob.median().values, 
                                    higher - data_var_where_eti_prob.median().values,
                                    ])
    etis_sec[key] = xr.Dataset(data_vars=data_vars, coords=coords, attrs=attrs)

In [ ]:
# Save to file
for prob_name in probs_names:
    etis_sec[prob_name].to_dataframe().transpose().to_csv(join(output_folders["tables"], f"eti{prob_name}_secpar{extension_outputs}.tsv"), sep="\t")

### Fitted values

In [ ]:
coords = {"fit": ["mean3_hdi", "delta_lower68mean3_hdi", "delta_higher68mean3_hdi",
                  "delta_lower99.7mean3_hdi", "delta_higher99.7mean3_hdi", 
                  "median", "delta_lower68median_eti", "delta_higher68median_eti",
                  "lower99.7_hdi", "higher99.7_hdi",
                  "lower99.7_eti", "higher99.7_eti"
                  ]}
attrs = {"description": "Infered values and errors with different criteria"}
data_vars = {}
for var in posterior_sec.keys():
    data_vars[var] = (["fit"], [hdis_sec["3%"][var].sel(hdi="mean_hdi").values, 
                                hdis_sec["68%"][var].sel(hdi="lower").values - hdis_sec["3%"][var].sel(hdi="mean_hdi").values,
                                hdis_sec["68%"][var].sel(hdi="higher").values - hdis_sec["3%"][var].sel(hdi="mean_hdi").values,
                                hdis_sec["99.7%"][var].sel(hdi="lower").values - hdis_sec["3%"][var].sel(hdi="mean_hdi").values,
                                hdis_sec["99.7%"][var].sel(hdi="higher").values - hdis_sec["3%"][var].sel(hdi="mean_hdi").values,
                                etis_sec["68%"][var].sel(eti="median").values,
                                etis_sec["68%"][var].sel(eti="lower").values - etis_sec["68%"][var].sel(eti="median").values,
                                etis_sec["68%"][var].sel(eti="higher").values - etis_sec["68%"][var].sel(eti="median").values,
                                hdis_sec["99.7%"][var].sel(hdi="lower").values, 
                                hdis_sec["99.7%"][var].sel(hdi="higher").values,
                                etis_sec["99.7%"][var].sel(eti="lower").values, 
                                etis_sec["99.7%"][var].sel(eti="higher").values,
                                ]
                      )
fitted_values_sec = xr.Dataset(data_vars=data_vars, coords=coords, attrs=attrs)
fitted_values_sec

In [ ]:
# Save to file
fitted_values_sec.to_dataframe().transpose().to_csv(join(output_folders["tables"], f"fittedvalues_secpar{extension_outputs}.tsv"), sep="\t")

## Corner plots

In [ ]:
thin_chain = 200
corner(posterior_sec.thin(indexers={"draw": thin_chain}), truths=[fitted_values_sec[var].sel(fit="mean3_hdi").values for var in fitted_values_sec])
pl.savefig(join(output_folders["plots"], f"corner_secpar_{extension_outputs}.png"))
pl.show()

## Posterior plots

#### Arviz posterior plots

In [ ]:
az.plot_posterior(posterior_sec, hdi_prob=0.68, point_estimate='mode')
pl.savefig(join(output_folders["plots"], f"posterior_secpar_arviz{extension_outputs}.png"))

### My posteriors

In [ ]:
best_value = "mean3_hdi"
shaded_lower1s = "delta_lower68mean3_hdi"
shaded_higher1s = "delta_higher68mean3_hdi"
shaded_lower3s = "delta_lower99.7mean3_hdi"
shaded_higher3s = "delta_higher99.7mean3_hdi"

# For multiple posterior plots
# nrows, ncols = az.plots.plot_utils.default_grid(n_items=len(infdata.posterior), grid=None, max_cols=4, min_cols=2)
# figsize, ax_labelsize, titlesize, xt_labelsize, linewidth, markersize = az.plots.plot_utils._scale_fig_size(figsize=(10,10), textsize=8, rows=1, cols=1)
# l_param = infdata.posterior.keys()

l_param = ["b_Frat", ]
paramname4label = {"b_Frat": "Eclipse Depth"}
factor = {"b_Frat": 1e6}
unit = {"b_Frat": "ppm"}

nrows = 1
ncols = len(l_param)
figsize = (4, 3)

fig, axes = pl.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, squeeze=False)
for ii, var in enumerate(l_param):
    i_col = ii % ncols
    i_row = ii // ncols
    ax = axes[i_row, i_col] 
    var_factor = factor[var] if var in factor else 1
    var_xlabel = paramname4label[var] if var in paramname4label else var
    if var in unit:
        var_xlabel += f" [{unit[var]}]"
    show_ylabel = True if i_col == 0 else False
    posterior_plot(samples=posterior_sec[var] * var_factor, 
                   values_wkwargs=[(fitted_values_sec[var].sel(fit=best_value).values * var_factor,{"color": "C0"})],
                   intervals_wkwargs=[(((fitted_values_sec[var].sel(fit=best_value).values + fitted_values_sec[var].sel(fit=shaded_lower3s).values) * var_factor, 
                                        (fitted_values_sec[var].sel(fit=best_value).values + fitted_values_sec[var].sel(fit=shaded_higher3s).values) * var_factor),
                                       {"color": "C0", 'alpha': 0.2}
                                       ),
                                      (((fitted_values_sec[var].sel(fit=best_value).values + fitted_values_sec[var].sel(fit=shaded_lower1s).values) * var_factor,  
                                        (fitted_values_sec[var].sel(fit=best_value).values + fitted_values_sec[var].sel(fit=shaded_higher1s).values) * var_factor),
                                       {"color": "C0", 'alpha': 0.5}
                                       ),
                                      ], 
                   xlabel=var_xlabel, 
                   show_ylabel=show_ylabel, ax=ax)

## Multi-modal PDF 

Omega is clearly multi-modal, let's try to derive multimodal HDIs

In [ ]:
hdis_multimode_sec = {}
coords = {"hdi": ["mean_hdi", "mean_all", "median_hdi", "median_all", "lower", "higher", 
                  "delta_lowermeanhdi", "delta_highermeanhdi", "delta_lowermeanall", "delta_highermeanall",
                  "delta_lowermedianhdi", "delta_higheredianhdi", "delta_lowermedianall", "delta_higheredianall",],
          }
attrs = {"description": "hdi interval caracterisation for multimodal distributions."}
l_param_names_multimodal = ["b_omega"]
max_modes = 2
for key, prob in zip(probs_names, probs_values):
    hdis_multimode_sec[key] = []
    hdi_prob = az.hdi(posterior_sec[l_param_names_multimodal], hdi_prob=prob, multimodal=True, max_modes=max_modes)
    for mode_i in range(hdi_prob.dims["mode"]):
        data_vars = {}
        for var in l_param_names_multimodal:
            if all(np.isnan(hdi_prob[var].sel(mode=mode_i))):
                print(f"WARNING: Both the upper and lower limit of the HDI are nan (prob={prob}, var={var})! So we cannot compute the quantities")
                l_values = [np.nan for col in coords["hdi"]]
            else:
                infdata_where_hdi_prob = posterior_sec[var].where(posterior_sec[var] > float(hdi_prob[var].sel(hdi="lower", mode=mode_i))).where(posterior_sec[var] < float(hdi_prob[var].sel(hdi="higher", mode=mode_i)), drop=True)
                l_values = [infdata_where_hdi_prob.mean().values, 
                            posterior_sec[var].mean().values,
                            infdata_where_hdi_prob.median().values,
                            posterior_sec[var].median().values,
                            float(hdi_prob[var].sel(hdi = "lower", mode=mode_i).values), 
                            float(hdi_prob[var].sel(hdi = "higher", mode=mode_i).values), 
                            float(hdi_prob[var].sel(hdi = "lower", mode=mode_i).values) - infdata_where_hdi_prob.mean().values, 
                            float(hdi_prob[var].sel(hdi = "higher", mode=mode_i).values) - infdata_where_hdi_prob.mean().values,
                            float(hdi_prob[var].sel(hdi = "lower", mode=mode_i).values) - posterior_sec[var].mean().values, 
                            float(hdi_prob[var].sel(hdi = "higher", mode=mode_i).values) - posterior_sec[var].mean().values,
                            float(hdi_prob[var].sel(hdi = "lower", mode=mode_i).values) - infdata_where_hdi_prob.median().values, 
                            float(hdi_prob[var].sel(hdi = "higher", mode=mode_i).values) - infdata_where_hdi_prob.median().values,
                            float(hdi_prob[var].sel(hdi = "lower", mode=mode_i).values) - posterior_sec[var].median().values, 
                            float(hdi_prob[var].sel(hdi = "higher", mode=mode_i).values) - posterior_sec[var].median().values,
                            ]
            data_vars[var] = (["hdi"], l_values)
        hdis_multimode_sec[key].append(xr.Dataset(data_vars=data_vars, coords=coords, attrs=attrs))

In [ ]:
# Save to file
for prob_name in probs_names:
    for i_mode, dataset_mode_i in enumerate(hdis_multimode_sec[prob_name]):
        dataset_mode_i.to_dataframe().transpose().to_csv(join(output_folders["tables"], f"hdi{prob_name}_mode{i_mode}_secpar{extension_outputs}.tsv"), sep="\t")